# 1. Saúde reprodutiva e planejamento familiar das mulheres

# 2. Membros
Antônio Alberto Pereira de Oliveira - 2021086504

Kauan Valente - 2023060464

Maria Eduarda Sampaio - 2022055831

# 3. Descrição dos dados

**URL**
https://www.kaggle.com/datasets/konradb/wellbeing-of-women-in-52-countries

Para reproduzir: baixar o dataset no Kaggle e colocar os arquivos `livwell.csv` e `indicators.csv` em `data/raw/`.

**Domínio**
Saúde reprodutiva e planejamento familiar das mulheres e sua relação com escolaridade, trabalho, autonomia nas decisões e violência doméstica. As consultas concentram-se em três áreas temáticas: Saúde reprodutiva, Educação e trabalho, e Autonomia e violência. Os demais indicadores selecionados (demografia, domicílio e condições de vida, saúde infantil e nutrição) permanecem no banco como contexto para análises futuras. Os indicadores vêm de pesquisas DHS (*Demographic and Health Surveys*) e permitem comparar países, regiões e períodos.

**Processamento**

O conjunto de dados LivWell é composto originalmente por 1.832 registros e 409 colunas (`livwell.csv`), além de um dicionário com 265 indicadores (`indicators.csv`). Cada registro corresponde a uma região de um país em uma pesquisa DHS, e cada indicador ocupa uma coluna (com uma coluna `_se` adicional para o erro-padrão, quando disponível).

O processamento, implementado nas células abaixo, segue as etapas:

1. **Seleção de indicadores** — foram selecionados manualmente 100 indicadores relacionados ao domínio do trabalho, distribuídos em 16 categorias. A seleção está em `data/processed/filtered_indicators.csv`.
2. **Filtragem** — de `livwell.csv` foram mantidas apenas as colunas de identificação (país, região, pesquisa, data da entrevista) e as colunas de valor e erro-padrão dos 100 indicadores. Nenhum registro foi removido.
3. **Normalização** — os dados foram decompostos em sete relações: `Pais`, `Regiao`, `Pesquisa`, `ColetaPesquisaRegiao`, `CategoriaIndicador`, `Indicador` e `Medicao`. As colunas de indicadores foram convertidas em linhas (uma medição por pesquisa × região × indicador), removendo apenas as medições sem valor observado e mantendo erros-padrão ausentes como nulos.
4. **Carga e verificação** — as relações foram inseridas em um banco SQLite com chaves primárias, estrangeiras e restrições de unicidade. A validação de chaves estrangeiras é desativada durante a carga (`PRAGMA foreign_keys = 0`) e a integridade referencial é verificada ao final.

**Ajustes nos dados originais**

- **País**: extraído como entidade própria. Além do nome, recebeu o continente, atribuído pelo grupo (o dataset não possui essa informação). Turquia e Armênia, países transcontinentais, foram classificados na Ásia.
- **Data da entrevista**: `interview_year_mean`, `interview_month_mean` e `CMC_interview_mean` variam entre as regiões de uma mesma pesquisa. Por isso pertencem ao relacionamento `ColetaPesquisaRegiao` (pesquisa × região), e não à pesquisa. Em `Pesquisa`, `CMC_interview_mean` é mantido como atributo derivado (média das regiões).
- **Categorias**: no dataset original a categoria é apenas um texto repetido em cada indicador. Ela foi extraída para a entidade `CategoriaIndicador`, enriquecida com o nome em português e uma área temática (agrupamento de categorias definido pelo grupo).
- **Tipo de indicador**: cada indicador recebeu um tipo (percentual, média, taxa, índice ou valor monetário), deduzido do sufixo do código e da descrição.
- **Nomes de região**: duas regiões aparecem com grafias diferentes entre pesquisas (`Hauncavelica`/`Huancavelica` no Peru e `Western  Cape`/`Western Cape` na África do Sul). Foi mantida a grafia da pesquisa mais recente.
- **Códigos de indicador**: alguns códigos usam hífen (`FF_ASFR_15-19`) enquanto a coluna correspondente no CSV usa ponto (`FF_ASFR_15.19`); o mapeamento é feito na filtragem.

## 3.1 Configuração

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

RAW = Path("data/raw")
PROCESSED = Path("data/processed")
DB_PATH = Path("saudefeminina.db")

# O banco é sempre recriado do zero a partir dos CSVs
DB_PATH.unlink(missing_ok=True)
con = sqlite3.connect(DB_PATH)
cur = con.cursor()

## 3.2 Leitura dos dados brutos

In [2]:
livwell = pd.read_csv(RAW / "livwell.csv")
indicadores_raw = pd.read_csv(RAW / "indicators.csv")
selecao = pd.read_csv(PROCESSED / "filtered_indicators.csv")

print("livwell.csv:", livwell.shape)
print("indicators.csv:", indicadores_raw.shape)
print("Indicadores selecionados:", len(selecao))

# Todo indicador selecionado deve existir no dicionário original
assert selecao["indicator_code"].isin(indicadores_raw["indicator_code"]).all()

selecao["indicator_category"].value_counts().to_frame("indicadores")

livwell.csv: (1832, 409)
indicators.csv: (265, 3)
Indicadores selecionados: 100


,indicadores
indicator_category,
Individual demographic information,13
Energy and information,12
Reproductive health and fertility,11
Decision power,10
Domestic Violence,10
Education,9
Fertility – complex indicators,9
Health – Birth level,5
Household characteristics – Womens level,4


## 3.3 Filtragem das colunas

In [3]:
COLUNAS_ID = [
    "country_name", "country_code", "year",
    "region_num_harmonized", "region_name_harmonized",
    "SurveyId", "interview_year_mean", "interview_month_mean", "CMC_interview_mean",
]

# Código do indicador -> coluna de valor (hífen vira ponto) e coluna de erro-padrão (se existir)
mapa = {}
for codigo in selecao["indicator_code"]:
    coluna = codigo.replace("-", ".")
    assert coluna in livwell.columns, f"Coluna não encontrada: {coluna}"
    coluna_se = coluna + "_se"
    mapa[codigo] = (coluna, coluna_se if coluna_se in livwell.columns else None)

colunas = COLUNAS_ID + [c for par in mapa.values() for c in par if c]
filtrado = livwell[colunas].copy()

print("Dados filtrados:", filtrado.shape)
print("Indicadores sem coluna de erro-padrão:", sum(se is None for _, se in mapa.values()))

Dados filtrados: (1832, 192)
Indicadores sem coluna de erro-padrão: 17


## 3.4 Criação do esquema

In [4]:
cur.executescript("""
PRAGMA foreign_keys = 0;

CREATE TABLE Pais (
    country_code TEXT PRIMARY KEY,
    country_name TEXT NOT NULL,
    continent    TEXT NOT NULL
);

-- Entidade fraca: region_num_harmonized só identifica a região dentro do país
CREATE TABLE Regiao (
    country_code           TEXT    NOT NULL REFERENCES Pais(country_code),
    region_num_harmonized  INTEGER NOT NULL,
    region_name_harmonized TEXT    NOT NULL,
    PRIMARY KEY (country_code, region_num_harmonized)
);

CREATE TABLE Pesquisa (
    SurveyId           TEXT PRIMARY KEY,
    country_code       TEXT    NOT NULL REFERENCES Pais(country_code),
    year               INTEGER NOT NULL,
    CMC_interview_mean REAL,
    UNIQUE (SurveyId, country_code)
);

-- Relacionamento M:N Pesquisa x Regiao. A FK composta para Pesquisa garante
-- que a região pertence ao mesmo país da pesquisa.
CREATE TABLE ColetaPesquisaRegiao (
    SurveyId              TEXT    NOT NULL,
    country_code          TEXT    NOT NULL,
    region_num_harmonized INTEGER NOT NULL,
    interview_year_mean   REAL,
    interview_month_mean  REAL,
    CMC_interview_mean    REAL,
    PRIMARY KEY (SurveyId, country_code, region_num_harmonized),
    FOREIGN KEY (SurveyId, country_code)
        REFERENCES Pesquisa(SurveyId, country_code),
    FOREIGN KEY (country_code, region_num_harmonized)
        REFERENCES Regiao(country_code, region_num_harmonized)
);

CREATE TABLE CategoriaIndicador (
    category_id      INTEGER PRIMARY KEY,
    category_name    TEXT NOT NULL UNIQUE,
    category_name_pt TEXT NOT NULL,
    category_area    TEXT NOT NULL
);

CREATE TABLE Indicador (
    indicator_code        TEXT PRIMARY KEY,
    indicator_description TEXT NOT NULL,
    indicator_type        TEXT NOT NULL,
    category_id           INTEGER NOT NULL REFERENCES CategoriaIndicador(category_id)
);

-- Relacionamento ternário Pesquisa x Regiao x Indicador
CREATE TABLE Medicao (
    measurement_id        INTEGER PRIMARY KEY,
    SurveyId              TEXT    NOT NULL REFERENCES Pesquisa(SurveyId),
    country_code          TEXT    NOT NULL,
    region_num_harmonized INTEGER NOT NULL,
    indicator_code        TEXT    NOT NULL REFERENCES Indicador(indicator_code),
    value                 REAL    NOT NULL,
    standard_error        REAL,
    FOREIGN KEY (country_code, region_num_harmonized)
        REFERENCES Regiao(country_code, region_num_harmonized),
    UNIQUE (SurveyId, country_code, region_num_harmonized, indicator_code)
);
""")
con.commit()

## 3.5 Inserção dos dados

### Pais, Regiao, Pesquisa e ColetaPesquisaRegiao

In [5]:
# Continente de cada país (atribuído pelo grupo)
CONTINENTES = {
    "África": ["BDI", "BEN", "BFA", "CIV", "CMR", "COD", "EGY", "ETH", "GAB", "GHA", "GIN",
               "KEN", "LBR", "LSO", "MAR", "MDG", "MLI", "MOZ", "MWI", "NAM", "NER", "NGA",
               "RWA", "SEN", "SLE", "TGO", "TZA", "UGA", "ZAF", "ZMB", "ZWE"],
    "América": ["BOL", "COL", "GTM", "HND", "HTI", "NIC", "PER"],
    "Ásia": ["ARM", "BGD", "IDN", "IND", "JOR", "KHM", "MDV", "NPL", "PAK", "PHL", "TJK",
             "TLS", "TUR", "VNM"],
}
continente_de = {pais: cont for cont, paises in CONTINENTES.items() for pais in paises}

pais = filtrado[["country_code", "country_name"]].drop_duplicates("country_code")
pais["continent"] = pais["country_code"].map(continente_de)
assert pais["continent"].notna().all()
pais = pais.sort_values("country_code")
pais.to_sql("Pais", con, if_exists="append", index=False)

# Regiao: para grafias divergentes, mantém o nome da pesquisa mais recente
regiao = (
    filtrado.sort_values("year", ascending=False)
    .drop_duplicates(["country_code", "region_num_harmonized"])
    [["country_code", "region_num_harmonized", "region_name_harmonized"]]
)
regiao.to_sql("Regiao", con, if_exists="append", index=False)

# Pesquisa: país e ano são únicos por pesquisa; CMC_interview_mean é a média das regiões
assert (filtrado.groupby("SurveyId")[["country_code", "year"]].nunique() == 1).all().all()
pesquisa = filtrado.groupby("SurveyId", as_index=False).agg(
    country_code=("country_code", "first"),
    year=("year", "first"),
    CMC_interview_mean=("CMC_interview_mean", "mean"),
)
pesquisa.to_sql("Pesquisa", con, if_exists="append", index=False)

# ColetaPesquisaRegiao: cada linha do CSV original é um par pesquisa x região
coleta = filtrado[[
    "SurveyId", "country_code", "region_num_harmonized",
    "interview_year_mean", "interview_month_mean", "CMC_interview_mean",
]]
coleta.to_sql("ColetaPesquisaRegiao", con, if_exists="append", index=False)

print("Países:", len(pais), "| Regiões:", len(regiao),
      "| Pesquisas:", len(pesquisa), "| Pares pesquisa x região:", len(coleta))

Países: 52 | Regiões: 481 | Pesquisas: 199 | Pares pesquisa x região: 1832


### CategoriaIndicador e Indicador

In [6]:
# Tradução e área temática de cada categoria (definidas pelo grupo)
CATEGORIAS = {
    "Decision power": ("Poder de decisão", "Autonomia e violência"),
    "Domestic Violence": ("Violência doméstica", "Autonomia e violência"),
    "Education": ("Educação", "Educação e trabalho"),
    "Work status": ("Situação de trabalho", "Educação e trabalho"),
    "Energy and information": ("Energia e informação", "Domicílio e condições de vida"),
    "Energy and information – per urban/rural area": ("Energia e informação – por área urbana/rural", "Domicílio e condições de vida"),
    "Household characteristics – Womens level": ("Características do domicílio – nível da mulher", "Domicílio e condições de vida"),
    "Household demographics – Household level": ("Demografia do domicílio – nível do domicílio", "Domicílio e condições de vida"),
    "Household Wealth": ("Riqueza do domicílio", "Domicílio e condições de vida"),
    "Socio-economic indicators": ("Indicadores socioeconômicos", "Domicílio e condições de vida"),
    "Individual demographic information": ("Informações demográficas individuais", "Demografia"),
    "Fertility – complex indicators": ("Fecundidade – indicadores compostos", "Saúde reprodutiva"),
    "Fertility preferences": ("Preferências de fecundidade", "Saúde reprodutiva"),
    "Reproductive health and fertility": ("Saúde reprodutiva e fecundidade", "Saúde reprodutiva"),
    "Health – Birth level": ("Saúde – nível do nascimento", "Saúde infantil e nutrição"),
    "Nutrition": ("Nutrição", "Saúde infantil e nutrição"),
}
assert set(selecao["indicator_category"]) == set(CATEGORIAS)

categoria = pd.DataFrame(
    [(nome, pt, area) for nome, (pt, area) in CATEGORIAS.items()],
    columns=["category_name", "category_name_pt", "category_area"],
)
categoria.insert(0, "category_id", range(1, len(categoria) + 1))
categoria.to_sql("CategoriaIndicador", con, if_exists="append", index=False)


def tipo_indicador(codigo):
    """Tipo de medida, deduzido do sufixo do código e da descrição do indicador."""
    if codigo.endswith("_p"):
        return "Percentual"
    if codigo.endswith("_mean"):
        return "Média"
    if codigo.startswith(("FF_", "HL_")):
        return "Taxa"
    if codigo == "gdp_pc":
        return "Valor monetário"
    return "Índice"  # WL_wealth_gini, WL_wealth_median, hdi


indicador = selecao.merge(categoria, left_on="indicator_category", right_on="category_name")
indicador["indicator_type"] = indicador["indicator_code"].map(tipo_indicador)
indicador = indicador[["indicator_code", "indicator_description", "indicator_type", "category_id"]]
indicador.to_sql("Indicador", con, if_exists="append", index=False)

print("Categorias:", len(categoria), "| Indicadores:", len(indicador))
indicador["indicator_type"].value_counts().to_frame("indicadores")

Categorias: 16 | Indicadores: 100


,indicadores
indicator_type,
Percentual,67
Média,15
Taxa,14
Índice,3
Valor monetário,1


### Medicao

Cada coluna de indicador vira um conjunto de linhas (formato largo → longo).

In [7]:
chaves = ["SurveyId", "country_code", "region_num_harmonized"]
partes = []
for codigo, (coluna, coluna_se) in mapa.items():
    parte = filtrado[chaves].copy()
    parte["indicator_code"] = codigo
    parte["value"] = filtrado[coluna]
    parte["standard_error"] = filtrado[coluna_se] if coluna_se else None
    partes.append(parte)

medicao = pd.concat(partes, ignore_index=True).dropna(subset=["value"])
medicao.to_sql("Medicao", con, if_exists="append", index=False)
con.commit()

print("Medições:", len(medicao))

Medições: 147829


## 3.6 Verificação

In [8]:
cur.execute("PRAGMA foreign_keys = 1")
erros_fk = cur.execute("PRAGMA foreign_key_check").fetchall()
print("Erros de integridade referencial:", erros_fk)
assert not erros_fk

# Toda medição corresponde a um par pesquisa x região existente em ColetaPesquisaRegiao
orfas = cur.execute("""
    SELECT COUNT(*) FROM Medicao m
    LEFT JOIN ColetaPesquisaRegiao a USING (SurveyId, country_code, region_num_harmonized)
    WHERE a.SurveyId IS NULL
""").fetchone()[0]
print("Medições sem par em ColetaPesquisaRegiao:", orfas)
assert orfas == 0

tabelas = ["Pais", "Regiao", "Pesquisa", "ColetaPesquisaRegiao", "CategoriaIndicador", "Indicador", "Medicao"]
pd.DataFrame(
    {"registros": [cur.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in tabelas]},
    index=tabelas,
)

Erros de integridade referencial: []
Medições sem par em ColetaPesquisaRegiao: 0


,registros
Pais,52
Regiao,481
Pesquisa,199
ColetaPesquisaRegiao,1832
CategoriaIndicador,16
Indicador,100
Medicao,147829


In [9]:
# Amostra do banco normalizado: medições com todas as entidades relacionadas
pd.read_sql_query("""
    SELECT pa.country_name, pa.continent, r.region_name_harmonized,
           p.SurveyId, p.year, a.interview_month_mean,
           c.category_name_pt, i.indicator_code, i.indicator_type,
           m.value, m.standard_error
    FROM Medicao m
    JOIN ColetaPesquisaRegiao a   USING (SurveyId, country_code, region_num_harmonized)
    JOIN Regiao r    USING (country_code, region_num_harmonized)
    JOIN Pais pa     USING (country_code)
    JOIN Pesquisa p  USING (SurveyId)
    JOIN Indicador i USING (indicator_code)
    JOIN CategoriaIndicador c USING (category_id)
    WHERE i.indicator_code = 'RH_contr_modern_p'
    LIMIT 5
""", con)

,country_name,continent,region_name_harmonized,SurveyId,year,interview_month_mean,category_name_pt,indicator_code,indicator_type,value,standard_error
0,Armenia,Ásia,Aragatsotn,AM2000DHS,2000,11.0,Saúde reprodutiva e fecundidade,RH_contr_modern_p,Percentual,10.12,0.97
1,Armenia,Ásia,Ararat,AM2000DHS,2000,11.0,Saúde reprodutiva e fecundidade,RH_contr_modern_p,Percentual,18.26,1.90
2,Armenia,Ásia,Armavir,AM2000DHS,2000,10.0,Saúde reprodutiva e fecundidade,RH_contr_modern_p,Percentual,12.53,1.11
3,Armenia,Ásia,Gegharkunik,AM2000DHS,2000,11.0,Saúde reprodutiva e fecundidade,RH_contr_modern_p,Percentual,13.09,1.96
4,Armenia,Ásia,Lori,AM2000DHS,2000,10.0,Saúde reprodutiva e fecundidade,RH_contr_modern_p,Percentual,14.43,1.56


# 4. Diagrama ER

# 5. Diagrama relacional

# 6. Consultas

## 6.1 Duas consultas envolvendo seleção e projeção

### 6.1.1 Consulta 1

### Consulta 1: maiores valores observados de uso de contraceptivos modernos

Seleção e projeção: lista as dez medições mais altas do indicador de uso atual de métodos contraceptivos modernos, com pesquisa, país e valor.

**Interpretação:** Os maiores percentuais observados mostram em quais pesquisas e países o uso de métodos modernos foi mais alto. Cada linha é uma medição regional; o ranking não representa uma média nacional.

In [ ]:
query = """
SELECT SurveyId, country_code, value
FROM Medicao
WHERE indicator_code = 'RH_contr_modern_p'
ORDER BY value DESC
LIMIT 10
"""
resultado = pd.read_sql_query(query, con)
resultado

SurveyId,country_code,value
CO2015DHS,COL,65.74
CO2015DHS,COL,65.22
CO2015DHS,COL,62.69
CO2015DHS,COL,61.53
ZA1998DHS,ZAF,60.36
CO2010DHS,COL,60.10
CO2015DHS,COL,60.04
CO2010DHS,COL,59.90
NM2006DHS,NAM,59.42
NM2013DHS,NAM,58.80


### 6.1.2 Consulta 2

### Consulta 2: regiões com taxa de fecundidade total igual ou superior a 4

Seleção e projeção: seleciona medições do indicador de fecundidade total (FF_TFR) acima ou iguais a 4 e exibe pesquisa, país e valor.

**Interpretação:** Os resultados identificam regiões de pesquisas com fecundidade total elevada. Como cada valor está associado a uma pesquisa e região, diferenças entre anos podem ser analisadas em consultas seguintes.

In [ ]:
query = """
SELECT SurveyId, country_code, value
FROM Medicao
WHERE indicator_code = 'FF_TFR' AND value >= 4
ORDER BY value DESC
LIMIT 10
"""
resultado = pd.read_sql_query(query, con)
resultado

SurveyId,country_code,value
NI1998DHS,NER,8.674
NI2012DHS,NER,8.355
CD2013DHS,COD,8.234
NI2012DHS,NER,8.210
TZ2010DHS,TZA,8.012
TZ1999DHS,TZA,7.989
NI2006DHS,NER,7.941
TZ1999DHS,TZA,7.920
UG2000DHS,UGA,7.863
NI2012DHS,NER,7.854


## 6.2 Três consultas envolvendo junção de duas relações

### 6.2.1 Consulta 3

### Consulta 3: medições da taxa de fecundidade total e tipo do indicador

Junção de duas relações: relaciona Medicao a Indicador para apresentar a descrição e o tipo de medida junto aos valores de FF_TFR.

**Interpretação:** A junção confirma que os valores pertencem ao indicador descrito como taxa de fecundidade total e facilita interpretar as medições sem depender apenas do código.

In [ ]:
query = """
SELECT m.SurveyId, m.country_code, m.region_num_harmonized,
       i.indicator_description, i.indicator_type, m.value
FROM Medicao AS m
JOIN Indicador AS i USING (indicator_code)
WHERE i.indicator_code = 'FF_TFR'
ORDER BY m.value DESC
LIMIT 10
"""
resultado = pd.read_sql_query(query, con)
resultado

SurveyId,country_code,region_num_harmonized,indicator_description,indicator_type,value
NI1998DHS,NER,4,Total Fertility Rate (based on a period of 3 year),Taxa,8.674
NI2012DHS,NER,4,Total Fertility Rate (based on a period of 3 year),Taxa,8.355
CD2013DHS,COD,5,Total Fertility Rate (based on a period of 3 year),Taxa,8.234
NI2012DHS,NER,2,Total Fertility Rate (based on a period of 3 year),Taxa,8.210
TZ2010DHS,TZA,15,Total Fertility Rate (based on a period of 3 year),Taxa,8.012
TZ1999DHS,TZA,15,Total Fertility Rate (based on a period of 3 year),Taxa,7.989
NI2006DHS,NER,4,Total Fertility Rate (based on a period of 3 year),Taxa,7.941
TZ1999DHS,TZA,16,Total Fertility Rate (based on a period of 3 year),Taxa,7.920
UG2000DHS,UGA,3,Total Fertility Rate (based on a period of 3 year),Taxa,7.863
NI2012DHS,NER,5,Total Fertility Rate (based on a period of 3 year),Taxa,7.854


### 6.2.2 Consulta 4

### Consulta 4: contracepção moderna por pesquisa e ano

Junção de duas relações: combina Medicao e Pesquisa para associar cada valor do indicador a seu ano de realização.

**Interpretação:** A consulta permite observar valores regionais de contracepção moderna ao longo do tempo e fornece a base para comparar pesquisas de anos diferentes.

In [ ]:
query = """
SELECT p.year, m.SurveyId, m.country_code, m.region_num_harmonized, m.value
FROM Medicao AS m
JOIN Pesquisa AS p USING (SurveyId)
WHERE m.indicator_code = 'RH_contr_modern_p'
ORDER BY p.year DESC, m.value DESC
LIMIT 10
"""
resultado = pd.read_sql_query(query, con)
resultado

year,SurveyId,country_code,region_num_harmonized,value
2019,SL2019DHS,SLE,3,25.78
2019,SL2019DHS,SLE,1,25.04
2019,SL2019DHS,SLE,4,24.90
2019,SL2019DHS,SLE,2,21.79
2019,SN2019DHS,SEN,4,21.62
2019,SN2019DHS,SEN,1,19.10
2019,SN2019DHS,SEN,2,16.39
2019,SN2019DHS,SEN,3,16.14
2018,ZM2018DHS,ZMB,5,36.45
2018,ZM2018DHS,ZMB,3,36.44


### 6.2.3 Consulta 5

### Consulta 5: indicadores de cada área temática

Junção de duas relações: associa cada indicador à sua categoria e à área temática definida pelo grupo.

**Interpretação:** O resultado mostra como os indicadores usados para estudar saúde reprodutiva se distribuem entre as categorias, além de explicitar o significado e o tipo de cada variável.

In [ ]:
query = """
SELECT c.category_area, c.category_name_pt, i.indicator_code,
       i.indicator_description, i.indicator_type
FROM Indicador AS i
JOIN CategoriaIndicador AS c USING (category_id)
WHERE c.category_area = 'Saúde reprodutiva'
ORDER BY c.category_name_pt, i.indicator_code
LIMIT 10
"""
resultado = pd.read_sql_query(query, con)
resultado

category_area,category_name_pt,indicator_code,indicator_description,indicator_type
Saúde reprodutiva,Fecundidade – indicadores compostos,FF_ASFR_15-19,Age Specific Fertility Rate (ASFR) for age period 15-19,Taxa
Saúde reprodutiva,Fecundidade – indicadores compostos,FF_ASFR_20-24,Age Specific Fertility Rate (ASFR) for age period 20-24,Taxa
Saúde reprodutiva,Fecundidade – indicadores compostos,FF_ASFR_25-29,Age Specific Fertility Rate (ASFR) for age period 25-29,Taxa
Saúde reprodutiva,Fecundidade – indicadores compostos,FF_ASFR_30-34,Age Specific Fertility Rate (ASFR) for age period 30-34,Taxa
Saúde reprodutiva,Fecundidade – indicadores compostos,FF_ASFR_35-39,Age Specific Fertility Rate (ASFR) for age period 35-39,Taxa
Saúde reprodutiva,Fecundidade – indicadores compostos,FF_ASFR_40-44,Age Specific Fertility Rate (ASFR) for age period 40-44,Taxa
Saúde reprodutiva,Fecundidade – indicadores compostos,FF_ASFR_45-49,Age Specific Fertility Rate (ASFR) for age period 45-49,Taxa
Saúde reprodutiva,Fecundidade – indicadores compostos,FF_GFR,"General Fertility Rate (GFR),",Taxa
Saúde reprodutiva,Fecundidade – indicadores compostos,FF_TFR,Total Fertility Rate (based on a period of 3 year),Taxa
Saúde reprodutiva,Preferências de fecundidade,FP_desired_daugh_mean,Average desired number of daugthers for all women,Média


## 6.3 Três consultas envolvendo junção de três ou mais relações

### 6.3.1 Consulta 6

### Consulta 6: países com maior uso de contracepção moderna

Junção de três relações: combina Medicao, Pesquisa e Pais para mostrar as maiores medições regionais com o nome do país e o ano.

**Interpretação:** Os registros permitem localizar geograficamente e temporalmente os maiores valores regionais, sem confundir uma medição de região com o resultado agregado do país.

In [ ]:
query = """
SELECT pa.country_name, p.year, m.SurveyId,
       m.region_num_harmonized, m.value AS percentual
FROM Medicao AS m
JOIN Pesquisa AS p USING (SurveyId)
JOIN Pais AS pa USING (country_code)
WHERE m.indicator_code = 'RH_contr_modern_p'
ORDER BY m.value DESC
LIMIT 10
"""
resultado = pd.read_sql_query(query, con)
resultado

country_name,year,SurveyId,region_num_harmonized,percentual
Colombia,2015,CO2015DHS,3,65.74
Colombia,2015,CO2015DHS,5,65.22
Colombia,2015,CO2015DHS,4,62.69
Colombia,2015,CO2015DHS,6,61.53
South Africa,1998,ZA1998DHS,1,60.36
Colombia,2010,CO2010DHS,3,60.10
Colombia,2015,CO2015DHS,2,60.04
Colombia,2010,CO2010DHS,5,59.90
Namibia,2006,NM2006DHS,2,59.42
Namibia,2013,NM2013DHS,4,58.80


### 6.3.2 Consulta 7

### Consulta 7: medições de indicadores de fecundidade por área temática

Junção de três relações: liga Medicao, Indicador e CategoriaIndicador para consultar valores de fecundidade com seus metadados.

**Interpretação:** A consulta compara as medições de dois indicadores de fecundidade, mantendo junto a descrição e o agrupamento temático para apoiar a interpretação.

In [ ]:
query = """
SELECT i.indicator_code, i.indicator_description,
       c.category_area, m.SurveyId, m.country_code, m.value
FROM Medicao AS m
JOIN Indicador AS i USING (indicator_code)
JOIN CategoriaIndicador AS c USING (category_id)
WHERE c.category_area = 'Saúde reprodutiva'
  AND i.indicator_code IN ('FF_TFR', 'FF_GFR')
ORDER BY i.indicator_code, m.value DESC
LIMIT 10
"""
resultado = pd.read_sql_query(query, con)
resultado

indicator_code,indicator_description,category_area,SurveyId,country_code,value
FF_GFR,"General Fertility Rate (GFR),",Saúde reprodutiva,NI1998DHS,NER,302.250
FF_GFR,"General Fertility Rate (GFR),",Saúde reprodutiva,NI2012DHS,NER,293.261
FF_GFR,"General Fertility Rate (GFR),",Saúde reprodutiva,NI2012DHS,NER,290.735
FF_GFR,"General Fertility Rate (GFR),",Saúde reprodutiva,NI2006DHS,NER,279.223
FF_GFR,"General Fertility Rate (GFR),",Saúde reprodutiva,CD2013DHS,COD,277.854
FF_GFR,"General Fertility Rate (GFR),",Saúde reprodutiva,TZ2010DHS,TZA,274.299
FF_GFR,"General Fertility Rate (GFR),",Saúde reprodutiva,TZ1999DHS,TZA,271.382
FF_GFR,"General Fertility Rate (GFR),",Saúde reprodutiva,NI1998DHS,NER,268.960
FF_GFR,"General Fertility Rate (GFR),",Saúde reprodutiva,UG1995DHS,UGA,268.861
FF_GFR,"General Fertility Rate (GFR),",Saúde reprodutiva,NI1992DHS,NER,267.957


### 6.3.3 Consulta 8

### Consulta 8: escolaridade e contracepção moderna nas mesmas regiões e pesquisas

Junção de três ou mais relações: relaciona duas cópias de Medicao pela mesma pesquisa e região, e acrescenta Pesquisa e Pais. Assim, cada linha compara escolaridade e contracepção no mesmo contexto.

**Interpretação:** A comparação pareada evita comparar regiões ou pesquisas diferentes. Ela permite explorar associação entre escolaridade secundária ou superior e contracepção moderna, mas não demonstra causalidade.

In [ ]:
query = """
SELECT pa.country_name, p.year, e.region_num_harmonized,
       e.value AS percentual_ensino_secundario,
       c.value AS percentual_contracepcao_moderna
FROM Medicao AS e
JOIN Medicao AS c
  ON c.SurveyId = e.SurveyId
 AND c.country_code = e.country_code
 AND c.region_num_harmonized = e.region_num_harmonized
JOIN Pesquisa AS p USING (SurveyId)
JOIN Pais AS pa ON pa.country_code = e.country_code
WHERE e.indicator_code = 'ED_attainment_secondary_higher_p'
  AND c.indicator_code = 'RH_contr_modern_p'
ORDER BY p.year DESC, pa.country_name
LIMIT 10
"""
resultado = pd.read_sql_query(query, con)
resultado

country_name,year,region_num_harmonized,percentual_ensino_secundario,percentual_contracepcao_moderna
Senegal,2019,1,43.81,19.10
Senegal,2019,2,21.52,16.39
Senegal,2019,3,22.86,16.14
Senegal,2019,4,35.29,21.62
Sierra Leone,2019,1,35.83,25.04
Sierra Leone,2019,2,32.17,21.79
Sierra Leone,2019,3,33.48,25.78
Sierra Leone,2019,4,64.70,24.90
Cameroon,2018,1,80.07,17.82
Cameroon,2018,2,18.39,6.61


## 6.4 Duas consultas envolvendo agregação sobre junção de duas ou mais relações

### 6.4.1 Consulta 9

### Consulta 9: média de contracepção moderna por continente

Agregação sobre junção de três relações: calcula média, menor e maior valor regional de contracepção moderna por continente.

**Interpretação:** As médias resumem a distribuição das medições regionais por continente; continentes com mais regiões ou pesquisas contribuem com mais observações, portanto a comparação é descritiva e ponderada pelo número de medições.

In [ ]:
query = """
SELECT pa.continent, COUNT(*) AS numero_medicoes,
       ROUND(AVG(m.value), 2) AS media_percentual,
       ROUND(MIN(m.value), 2) AS minimo,
       ROUND(MAX(m.value), 2) AS maximo
FROM Medicao AS m
JOIN Pais AS pa USING (country_code)
JOIN Indicador AS i USING (indicator_code)
WHERE i.indicator_code = 'RH_contr_modern_p'
GROUP BY pa.continent
ORDER BY media_percentual DESC
"""
resultado = pd.read_sql_query(query, con)
resultado

continent,numero_medicoes,media_percentual,minimo,maximo
América,374,32.92,6.95,65.74
Ásia,579,27.33,1.41,55.38
África,871,18.93,0.12,60.36


### 6.4.2 Consulta 10

### Consulta 10: evolução anual da fecundidade total

Agregação sobre junção de três relações: calcula a média, o mínimo, o máximo e o número de medições regionais de FF_TFR por ano.

**Interpretação:** A série resume a fecundidade total observada em cada ano. O número de medições varia entre anos e a média é regional, então deve ser interpretada como panorama descritivo do conjunto disponível.

In [ ]:
query = """
SELECT p.year, COUNT(*) AS numero_medicoes,
       ROUND(AVG(m.value), 2) AS media_tfr,
       ROUND(MIN(m.value), 2) AS minimo,
       ROUND(MAX(m.value), 2) AS maximo
FROM Medicao AS m
JOIN Pesquisa AS p USING (SurveyId)
JOIN Indicador AS i USING (indicator_code)
WHERE i.indicator_code = 'FF_TFR'
GROUP BY p.year
ORDER BY p.year
"""
resultado = pd.read_sql_query(query, con)
resultado

year,numero_medicoes,media_tfr,minimo,maximo
1990,5,2.83,2.45,3.58
1991,4,4.83,4.51,5.13
1992,32,6.00,3.38,7.85
1993,45,4.34,2.36,6.55
1994,52,3.85,1.79,6.04
1995,20,4.80,2.52,7.38
1996,68,5.22,2.53,7.68
1997,63,3.89,1.84,7.47
1998,58,4.64,2.19,8.67
1999,61,4.11,1.77,7.99


# 7. Autoavaliação dos membros